## ForestCasting using sklearn models.

Time Series Forecasting usually does not require complex neural nets or Arima. Arima is very good for forecasting time series data, but, more powerful ML algorithms along the lines of Xgboost can be used instead.

## Step 01

Fetch data. Select a valid series id and have fred api key for access to the data.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.data import fetch_data

# Fed Funds Series ID
SERIES_ID = "FEDFUNDS"

df = fetch_data(SERIES_ID).set_index("Date")

## Step 02: Split data
Does not need to be scaled with XGBoost

In [2]:
# build lag features: predict this month's rate from the previous N months
N_LAGS = 12

features = df.copy()
for lag in range(1, N_LAGS + 1):
    features[f"lag_{lag}"] = features[SERIES_ID].shift(lag)
features = features.dropna()

X = features.drop(SERIES_ID, axis=1)
y = features[SERIES_ID]

# chronological split -- shuffling would leak future rates into training
split = int(len(features) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
X_train.shape, X_test.shape

((684, 12), (171, 12))

## Forecast using scikit-learn models.

### Models Selected for Forecasting**

- **Linear**
- **XGBoost**

Choosing one linear model (`Linear Regression`) to see how it performs compared to ensemble model.

In [3]:
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

from src.metrics import get_results

models = {
    "XGBRegressor": XGBRegressor(n_estimators=100),
    "LinearRegression": LinearRegression(),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"{name}: {get_results(y_test, y_pred)}")

XGBRegressor: {'r2': 0.9821317781923856, 'mape': 0.32504790008319007, 'rmse': np.float64(0.25126769841293967), 'mae': 0.1469510845395557}
LinearRegression: {'r2': 0.9947068940147646, 'mape': 0.3413612651010759, 'rmse': np.float64(0.13675754288085826), 'mae': 0.08314535006688309}
